In [0]:
dbutils.widgets.text("catalog", "vp_ce_2026_07_1382", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
-- 1. Entitlements table — one row per user
CREATE TABLE IF NOT EXISTS ops.user_entitlements (
  user_email   STRING,
  allowed_city STRING,
  can_view_pii BOOLEAN
);

-- 2. Row-filter UDF: keeps only rows whose city matches the current user's allowed_city
CREATE OR REPLACE FUNCTION ops.rf_allowed_city (city STRING)
RETURNS BOOLEAN
RETURN EXISTS (
  SELECT 1
  FROM ops.user_entitlements e
  WHERE e.user_email  = current_user()
    AND e.allowed_city = city
);

-- 3. Column-mask UDF: reveals the real value only when the current user's can_view_pii is TRUE
CREATE OR REPLACE FUNCTION ops.cm_pii (val STRING)
RETURNS STRING
RETURN CASE
  WHEN EXISTS (
    SELECT 1
    FROM ops.user_entitlements e
    WHERE e.user_email   = current_user()
      AND e.can_view_pii = TRUE
  )
  THEN val
  ELSE '***REDACTED***'
END;

-- 4. Apply the row filter and column mask to the entitlements table itself
--    so a user can only see their own row, and PII columns on other tables
--    are governed by the same entitlements lookup.
ALTER TABLE ops.user_entitlements
  SET ROW FILTER ops.rf_allowed_city (allowed_city);

-- Example: apply to a data table
-- ALTER TABLE my_catalog.my_schema.customers
--   SET ROW FILTER ops.rf_allowed_city (city);
-- ALTER TABLE my_catalog.my_schema.customers
--   ALTER COLUMN ssn SET MASK ops.cm_pii (ssn);